In [72]:
using JuMP, HiGHS, LinearAlgebra, MathOptInterface

4×4 Matrix{Bool}:
 0  0  0  1
 0  0  1  0
 0  1  0  0
 1  0  0  0

In [157]:
m2 = Model(HiGHS.Optimizer)
n = 4
P = Matrix(I,n,n)[n:-1:1, :]
@variable(m2, x[1:n,1:n], Bin)
@constraint(m2, c1, mapslices(sum, x, dims = [1]) .<= ones(1,n))
@constraint(m2, c2, mapslices(sum, x, dims = [2]) .<= ones(n,1))
@constraint(m2, c3, [sum(diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@constraint(m2, c4, [sum(P*diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@objective(m2, Max, sum([x[i,j] for i=1:n, j=1:n]))

x[1,1] + x[2,1] + x[3,1] + x[4,1] + x[1,2] + x[2,2] + x[3,2] + x[4,2] + x[1,3] + x[2,3] + x[3,3] + x[4,3] + x[1,4] + x[2,4] + x[3,4] + x[4,4]

In [158]:
print(m2)

Max x[1,1] + x[2,1] + x[3,1] + x[4,1] + x[1,2] + x[2,2] + x[3,2] + x[4,2] + x[1,3] + x[2,3] + x[3,3] + x[4,3] + x[1,4] + x[2,4] + x[3,4] + x[4,4]
Subject to
 c1 : x[1,1] + x[2,1] + x[3,1] + x[4,1] ≤ 1
 c1 : x[1,2] + x[2,2] + x[3,2] + x[4,2] ≤ 1
 c1 : x[1,3] + x[2,3] + x[3,3] + x[4,3] ≤ 1
 c1 : x[1,4] + x[2,4] + x[3,4] + x[4,4] ≤ 1
 c2 : x[1,1] + x[1,2] + x[1,3] + x[1,4] ≤ 1
 c2 : x[2,1] + x[2,2] + x[2,3] + x[2,4] ≤ 1
 c2 : x[3,1] + x[3,2] + x[3,3] + x[3,4] ≤ 1
 c2 : x[4,1] + x[4,2] + x[4,3] + x[4,4] ≤ 1
 c3 : x[3,1] + x[4,2] ≤ 1
 c3 : x[2,1] + x[3,2] + x[4,3] ≤ 1
 c3 : x[1,1] + x[2,2] + x[3,3] + x[4,4] ≤ 1
 c3 : x[1,2] + x[2,3] + x[3,4] ≤ 1
 c3 : x[1,3] + x[2,4] ≤ 1
 c4 : x[2,1] + x[1,2] ≤ 1
 c4 : x[3,1] + x[2,2] + x[1,3] ≤ 1
 c4 : x[4,1] + x[3,2] + x[2,3] + x[1,4] ≤ 1
 c4 : x[4,2] + x[3,3] + x[2,4] ≤ 1
 c4 : x[4,3] + x[3,4] ≤ 1
 x[1,1] binary
 x[2,1] binary
 x[3,1] binary
 x[4,1] binary
 x[1,2] binary
 x[2,2] binary
 x[3,2] binary
 x[4,2] binary
 x[1,3] binary
 x[2,3] binary
 x[3,3] b

In [159]:
set_silent(m2)
optimize!(m2)
is_solved_and_feasible(m2)

true

In [160]:
round.(Int, value.(x))

4×4 Matrix{Int64}:
 0  0  1  0
 1  0  0  0
 0  0  0  1
 0  1  0  0

In [135]:
objective_value(m2)

5.013148258832215

In [136]:
dual.(c1), dual.(c2), dual.(c3), dual.(c4)

([0.0 0.0 … 0.0 0.0], [0.0; 0.0; … ; 0.0; 0.0;;], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])

In [137]:
get_attribute.(x, MOI.VariableBasisStatus())

5×5 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  …  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2     NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2     NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2     NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2     NONBASIC_AT_LOWER::BasisStatusCode = 2

In [140]:
get_attribute.(c1, MOI.ConstraintBasisStatus())

1×5 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1

In [70]:
solution_summary(m2; verbose = true)

solution_summary(; result = 1, verbose = true)
├ solver_name          : HiGHS
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ ├ raw_status         : kHighsModelStatusOptimal
│ └ objective_bound    : 4.00000e+00
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : NO_SOLUTION
│ ├ objective_value      : 4.00000e+00
│ ├ dual_objective_value : NaN
│ ├ relative_gap         : 0.00000e+00
│ └ value
│   ├ x[1,1] : -0.00000e+00
│   ├ x[1,2] : 0.00000e+00
│   ├ x[1,3] : 1.00000e+00
│   ├ x[1,4] : 0.00000e+00
│   ├ x[2,1] : 1.00000e+00
│   ├ x[2,2] : 0.00000e+00
│   ├ x[2,3] : 0.00000e+00
│   ├ x[2,4] : -0.00000e+00
│   ├ x[3,1] : -0.00000e+00
│   ├ x[3,2] : 0.00000e+00
│   ├ x[3,3] : 0.00000e+00
│   ├ x[3,4] : 1.00000e+00
│   ├ x[4,1] : 0.00000e+00
│   ├ x[4,2] : 1.00000e+00
│   ├ x[4,3] : -0.00000e+00
│   └ x[4,4] : -0.00000e+00
└ Work counters
  ├ solve_time (sec)   : 1.19279e-02
  ├ simplex_iterations : 13
  ├ barrier_iterati

In [71]:
get_attribute.(x, MOI.VariableBasisStatus())

4×4 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2

In [95]:
# ## A more complicated example

# Often, you may want to work with the basis of a model that is not in a nice
# standard form. For example:

model = Model(HiGHS.Optimizer)
set_silent(model)
@variable(model, x >= 0)
@variable(model, 0 <= y <= 3)
@variable(model, z <= 1)
@objective(model, Min, 12x + 20y - z)
@constraint(model, c1, 6x + 8y >= 100)
@constraint(model, c2, 7x + 12y >= 120)
@constraint(model, c3, x + y <= 20)
optimize!(model)
assert_is_solved_and_feasible(model)

# A common way to query the basis status of every variable is:

v_basis = Dict(
    xi => get_attribute(xi, MOI.VariableBasisStatus()) for
    xi in all_variables(model)
)

# Despite the model having three constraints, there are only two basic
# variables. Since the basis matrix must be square, where is the other basic
# variable?

# The answer is that solvers will reformulate inequality constraints:
# ```math
# A x \le b
# ```
# into the system:
# ```math
# A x + Is = b
# ```
# Thus, for every inequality constraint there is a slack variable `s`.

# Query the basis status of the slack variables associated with a constraint
# using [`MOI.ConstraintBasisStatus`](@ref):

c_basis = Dict(
    ci => get_attribute(ci, MOI.ConstraintBasisStatus()) for ci in
    all_constraints(model; include_variable_in_set_constraints = false)
)

# Thus, the basis is formed by `x`, `y`, and the slack associated with `c3`.

# A simple way to get the `A` matrix of an unstructured linear program is with
# [`lp_matrix_data`](@ref):

matrix = lp_matrix_data(model)
matrix.A

# You can check the permutation of the rows and columns using

matrix.variables

# and

matrix.affine_constraints

# We can construct the slack column associated with `c3` as:

s_column = zeros(size(matrix.A, 1))
s_column[3] = 1.0

# The full basis matrix is therefore:

B = hcat(matrix.A[:, [1, 2]], s_column)

# [`lp_matrix_data`](@ref) returns separate vectors for the lower and upper row
# bounds. Convert to a single right-hand side vector by taking the finite
# elements:

b = ifelse.(isfinite.(matrix.b_lower), matrix.b_lower, matrix.b_upper)

# Solving the Basis system as before yields:

B \ b

# which is the value of `x`, `y`, and the slack associated with `c3`.

# ## Identifying degenerate variables

# Another common task is identifying degenerate variables. A degenerate variable
# is a basic variable that has an optimal value at its lower or upper bound.

# Here is a function that computes whether a variable is degenerate:

function is_degenerate(x)
    if get_attribute(x, MOI.VariableBasisStatus()) == MOI.BASIC
        return (has_lower_bound(x) && ≈(value(x), lower_bound(x))) ||
               (has_upper_bound(x) && ≈(value(x), upper_bound(x)))
    end
    return false
end

# A simple example of a linear program with a degenerate solution is:

A, b, c = [1 1; 0 1], [1, 1], [1, 1]
model = Model(HiGHS.Optimizer);
set_silent(model)
@variable(model, x[1:2] >= 0)
@objective(model, Min, c' * x)
@constraint(model, A * x == b)
optimize!(model)
degenerate_variables = filter(is_degenerate, all_variables(model))

# The solution is degenerate because:

value(x[1])

# and

get_attribute(x[1], MOI.VariableBasisStatus())

BASIC::BasisStatusCode = 0

In [96]:
v_basis

Dict{VariableRef, MathOptInterface.BasisStatusCode} with 3 entries:
  y => BASIC
  x => BASIC
  z => NONBASIC_AT_UPPER

In [97]:
c_basis

Dict{ConstraintRef{Model, C, ScalarShape} where C, MathOptInterface.BasisStatusCode} with 3 entries:
  c2 : 7 x + 12 y ≥ 120 => NONBASIC
  c3 : x + y ≤ 20       => BASIC
  c1 : 6 x + 8 y ≥ 100  => NONBASIC

In [103]:
value(x),value(y),value(z)

([-0.0, 1.0], 1.25, 1.0)

In [104]:

solution_summary(model; verbose = true)

solution_summary(; result = 1, verbose = true)
├ solver_name          : HiGHS
├ Termination
│ ├ termination_status : OPTIMAL
│ ├ result_count       : 1
│ ├ raw_status         : kHighsModelStatusOptimal
│ └ objective_bound    : 1.00000e+00
├ Solution (result = 1)
│ ├ primal_status        : FEASIBLE_POINT
│ ├ dual_status          : FEASIBLE_POINT
│ ├ objective_value      : 1.00000e+00
│ ├ dual_objective_value : 1.00000e+00
│ ├ relative_gap         : 0.00000e+00
│ └ value
│   ├ x[1] : -0.00000e+00
│   └ x[2] : 1.00000e+00
└ Work counters
  ├ solve_time (sec)   : 2.00459e-04
  ├ simplex_iterations : 0
  ├ barrier_iterations : 0
  └ node_count         : -1

In [162]:
q3 = Model(HiGHS.Optimizer)
n = 3
P = Matrix(I,n,n)[n:-1:1, :]
@variable(q3, x[1:n,1:n], Bin)
@constraint(q3, c1, mapslices(sum, x, dims = [1]) .<= ones(1,n))
@constraint(q3, c2, mapslices(sum, x, dims = [2]) .<= ones(n,1))
@constraint(q3, c3, [sum(diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@constraint(q3, c4, [sum(P*diagm(i=>ones(n-abs(i))).*x) for i=-(n-2):(n-2)] .<= ones(2n-3))
@objective(q3, Max, sum([x[i,j] for i=1:n, j=1:n]))
print(q3)

Max x[1,1] + x[2,1] + x[3,1] + x[1,2] + x[2,2] + x[3,2] + x[1,3] + x[2,3] + x[3,3]
Subject to
 c1 : x[1,1] + x[2,1] + x[3,1] ≤ 1
 c1 : x[1,2] + x[2,2] + x[3,2] ≤ 1
 c1 : x[1,3] + x[2,3] + x[3,3] ≤ 1
 c2 : x[1,1] + x[1,2] + x[1,3] ≤ 1
 c2 : x[2,1] + x[2,2] + x[2,3] ≤ 1
 c2 : x[3,1] + x[3,2] + x[3,3] ≤ 1
 c3 : x[2,1] + x[3,2] ≤ 1
 c3 : x[1,1] + x[2,2] + x[3,3] ≤ 1
 c3 : x[1,2] + x[2,3] ≤ 1
 c4 : x[2,1] + x[1,2] ≤ 1
 c4 : x[3,1] + x[2,2] + x[1,3] ≤ 1
 c4 : x[3,2] + x[2,3] ≤ 1
 x[1,1] binary
 x[2,1] binary
 x[3,1] binary
 x[1,2] binary
 x[2,2] binary
 x[3,2] binary
 x[1,3] binary
 x[2,3] binary
 x[3,3] binary


In [ ]:
set_silent(q3)
optimize!(q3)
is_solved_and_feasible(q3)

true

In [165]:
round.(Int,value.(x))

3×3 Matrix{Int64}:
 0  0  0
 1  0  0
 0  0  1

In [166]:
get_attribute.(c1, MOI.ConstraintBasisStatus())

1×3 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1  NONBASIC::BasisStatusCode = 1

In [167]:
get_attribute.(c2, MOI.ConstraintBasisStatus())

3×1 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1

In [168]:
get_attribute.(c3, MOI.ConstraintBasisStatus())

3-element Vector{MathOptInterface.BasisStatusCode}:
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1

In [169]:
get_attribute.(c4, MOI.ConstraintBasisStatus())

3-element Vector{MathOptInterface.BasisStatusCode}:
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1
 NONBASIC::BasisStatusCode = 1

In [170]:
get_attribute.(x, MOI.VariableBasisStatus())

3×3 Matrix{MathOptInterface.BasisStatusCode}:
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2
 NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2  NONBASIC_AT_LOWER::BasisStatusCode = 2

In [171]:
dual.(c1)

1×3 Matrix{Float64}:
 0.0  0.0  0.0

In [172]:
dual.(c2)

3×1 Matrix{Float64}:
 0.0
 0.0
 0.0

In [173]:
dual.(c3)

3-element Vector{Float64}:
 0.0
 0.0
 0.0

In [174]:
dual.(c4)

3-element Vector{Float64}:
 0.0
 0.0
 0.0